# BÀI TẬP LỚN 02 – HỌC SÂU: NHẬN DIỆN CẢM XÚC KHUÔN MẶT BẰNG MẠNG CNN (FER2013)

- **Lớp:** DCT123C4 | **GVHD:** ThS. Đỗ Như Tài
- **Nhóm thực hiện:** Nguyễn Đình Cường (3123411043 – TV1, Nhóm trưởng), Châu Quốc Bảo (3123411024 – TV2), Nguyễn Hoàng Long (3123411179 – TV3), Võ Huỳnh Minh Sang (3123411256 – TV4).
- **Mục đích notebook:** Chạy tái lập toàn bộ pipeline PyTorch từ khám phá dữ liệu (EDA), huấn luyện Nhóm 1 & Nhóm 2, thẩm định Best Model (có/không TTA trên `PublicTest`), đánh giá ngoại kiểm một lần duy nhất trên `PrivateTest`, đến trực quan hóa Final Convolution Filters, Feature Maps và Grad-CAM.

In [ ]:
import os
from pathlib import Path
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

DATA_PATH = Path('data/fer2013.csv')
assert DATA_PATH.is_file(), 'Vui long dat fer2013.csv vao data/fer2013.csv theo huong dan trong data/README_DATA.md'

## 1. Kiểm thử đơn vị (Unit Tests) & Khám phá dữ liệu (EDA – TV1)

In [ ]:
!python -m unittest discover -s tests -v
!python -m src.eda data/fer2013.csv
!python -m src.models

## 2. Thực nghiệm Nhóm 1 (Kiến trúc CNN + Fail Case VGG16 – TV2) & Nhóm 2 (Ablation – TV3)

Bỏ chú thích hai lệnh bên dưới nếu muốn huấn luyện lại toàn bộ 17 thực nghiệm từ đầu trên GPU.

In [ ]:
# !python -m src.run_exp1_architectures --data-path data/fer2013.csv --max-epochs 60 --patience 12 --batch-size 64 --seed 42
# !python -m src.run_exp2_training_techniques --data-path data/fer2013.csv --max-epochs 60 --patience 12 --batch-size 64 --seed 42
import pandas as pd
pd.read_csv('logs/experiment_runs.csv')[['exp_id', 'model_name', 'activation', 'optimizer', 'aug_mode', 'loss_type', 'val_loss', 'val_acc', 'val_macro_f1']]

## 3. Đánh giá Best Model trên PublicTest (Standard vs 3-View TTA) & PrivateTest (TV4 – EXP-18)

Có thể bổ sung `--train-seeds 42 100 2024` khi chạy trên GPU để huấn luyện lại cấu hình tổ hợp qua 3 seed; mặc định đánh giá trực tiếp checkpoint đã kiểm chứng và khóa chiến lược TTA từ `PublicTest` trước khi đánh giá một lần duy nhất trên `PrivateTest`.

In [ ]:
!python -m src.evaluate_best_model --data-path data/fer2013.csv --checkpoint checkpoints/EXP-05_ImprovedFERNet.pth --batch-size 128
print('--- PrivateTest Summary ---')
display(pd.read_csv('logs/best_model_private_metrics.csv'))
print('--- Per-Class Metrics on PrivateTest ---')
display(pd.read_csv('logs/best_model_per_class_metrics.csv'))

## 4. Trực quan hóa Final Convolution Filters, Feature Maps & Grad-CAM (TV4)

In [ ]:
!python -m src.visualize_filters --checkpoint checkpoints/best_model.pth --data-path data/fer2013.csv --split PrivateTest
from IPython.display import Image, display
for fig_name in [
    'figures/viz_first_conv_filters.png',
    'figures/viz_final_conv_filters_weights.png',
    'figures/viz_final_conv_feature_maps.png',
    'figures/viz_gradcam_7emotions.png',
    'figures/best_confusion_matrix.png',
    'figures/best_true_cases.png',
    'figures/best_wrong_cases.png',
]:
    if Path(fig_name).is_file():
        print(fig_name)
        display(Image(filename=fig_name, width=720))